# 🏭 Voice Worker — โรงงานเสียงพากย์ประจำช่อง (OmniVoice-Thai)

**หน้าที่:** อ่านไฟล์งานทั้งหมดใน `tts-ai/voice-jobs/*.json` จาก repo → เจนเสียงทุก segment ด้วยเสียงโคลนประจำช่อง (ตาม `voice_profile.json`) → ได้ zip เสียง + `timing.json` ไปวางในโปรเจกต์คลิป

**วงจรการทำงานข้ามโปรเจกต์:**
1. Claude (เซสชันไหนก็ได้) เขียนสคริปต์คลิป → สร้าง `tts-ai/voice-jobs/<ชื่องาน>.json` → commit
2. เจ้าของช่องเปิด notebook นี้ → Runtime **T4 GPU** → **Run all** (~10 นาที)
3. ได้ไฟล์ zip → แตกใส่โฟลเดอร์โปรเจกต์บน PC → ทำ CapCut ต่อ

ทุกข้อความผ่านตัวแปลงตัวเลข `normalize_th` อัตโนมัติ

In [ ]:
# 1) ติดตั้ง (~2 นาที)
!pip install -q omnivoice edge-tts soundfile pythainlp
import torch
print('GPU:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'ไม่มี — แนะนำเปิด T4')

In [ ]:
# 2) ดึง repo ทั้งก้อน (ได้ทั้งโค้ด, โปรไฟล์เสียง, ไฟล์เสียงอ้างอิง และงานทั้งหมด)
import glob, json, os, sys, time

BRANCH = 'claude/tts-ai-tools-resources-m5rku5'
!rm -rf repo.zip claude-project1-*
!wget -q -O repo.zip https://codeload.github.com/suthipongs10-a11y/claude-project1/zip/refs/heads/{BRANCH}
!unzip -q repo.zip

TTS_DIR = glob.glob('claude-project1-*/tts-ai')[0]
sys.path.insert(0, TTS_DIR)
from normalize_th import normalize_th

PROFILE = json.load(open(f'{TTS_DIR}/voice_profile.json', encoding='utf-8'))
JOB_FILES = sorted(glob.glob(f'{TTS_DIR}/voice-jobs/*.json'))
print('เวอร์ชันเสียง:', PROFILE['voice_version'])
print('พบงาน', len(JOB_FILES), 'ไฟล์:', [os.path.basename(j) for j in JOB_FILES])

In [ ]:
# 3) เสียงอ้างอิงประจำช่อง (จาก voice_profile) — ยังไม่มีเสียงจริงจะใช้เปรมวดีชั่วคราว
ref_audio = f"{TTS_DIR}/{PROFILE['ref_audio']}"
ref_text_file = f"{TTS_DIR}/{PROFILE['ref_text_file']}"

if os.path.exists(ref_audio) and os.path.exists(ref_text_file):
    REF_TEXT = open(ref_text_file, encoding='utf-8').read().strip()
    src = f"เสียงจริงประจำช่อง ({PROFILE['voice_version']}) ✅"
else:
    import edge_tts
    REF_TEXT = 'สวัสดีครับ ยินดีต้อนรับเข้าสู่ช่องของเรา วันนี้มีเรื่องราวน่าสนใจมาเล่าให้ฟังกันครับ'
    await edge_tts.Communicate(REF_TEXT, 'th-TH-PremwadeeNeural').save('ref_tmp.mp3')
    ref_audio, src = 'ref_tmp.mp3', '⚠️ เปรมวดีชั่วคราว (ยังไม่มีเสียงจริงใน repo)'

!ffmpeg -y -loglevel error -i {ref_audio} -ar 24000 -ac 1 ref_24k.wav
REF_AUDIO = 'ref_24k.wav'
print('เสียงอ้างอิง:', src)

In [ ]:
# 4) โหลดโมเดล (~3-4 นาทีครั้งแรก)
import soundfile as sf
from omnivoice import OmniVoice

dev = 'cuda:0' if torch.cuda.is_available() else 'cpu'
dt = torch.float16 if torch.cuda.is_available() else torch.float32
try:
    model = OmniVoice.from_pretrained(PROFILE['model_id'], device_map=dev, dtype=dt)
    print('โมเดล:', PROFILE['model_id'])
except Exception as e:
    print('โหลดตัวหลักไม่ได้ ->', e)
    model = OmniVoice.from_pretrained(PROFILE['fallback_model_id'], device_map=dev, dtype=dt)
    print('โมเดล:', PROFILE['fallback_model_id'])

In [ ]:
# 5) เจนเสียงทุกงาน ทุก segment + timing.json ต่องาน
SR = PROFILE.get('sample_rate', 24000)

for jf in JOB_FILES:
    job = json.load(open(jf, encoding='utf-8'))
    name = job.get('job') or os.path.basename(jf).replace('.json', '')
    outdir = f'voice_out/{name}'
    os.makedirs(outdir, exist_ok=True)
    timing = []
    print(f'\n=== งาน: {name} ({len(job["segments"])} segments) ===')
    for seg in job['segments']:
        sid, text = seg['id'], seg['text']
        try:
            gen_text = normalize_th(text) if PROFILE.get('normalize_numbers', True) else text
            t0 = time.time()
            audio = model.generate(text=gen_text, ref_audio=REF_AUDIO, ref_text=REF_TEXT)
            f = f'{outdir}/{sid}.wav'
            sf.write(f, audio[0], SR)
            secs = round(len(audio[0]) / SR, 3)
            timing.append({'id': sid, 'file': f'{sid}.wav', 'seconds': secs,
                           'chars': len(text), 'text': text, 'gen_text': gen_text})
            print(f'  เสร็จ: {sid} -> เสียงยาว {secs:.1f} วิ (เจน {time.time()-t0:.1f} วิ)')
        except Exception as e:
            print(f'  ข้าม {sid} ->', e)
    json.dump(timing, open(f'{outdir}/timing.json', 'w', encoding='utf-8'),
              ensure_ascii=False, indent=2)
    print(f'  รวมเสียง {sum(t["seconds"] for t in timing):.1f} วิ -> {outdir}/timing.json')

In [ ]:
# 6) 🎧 ฟังตรวจก่อนโหลด
from IPython.display import Audio, display, Markdown
display(Markdown(f'### เสียงอ้างอิง ({src})'))
display(Audio(REF_AUDIO))
for tj in sorted(glob.glob('voice_out/*/timing.json')):
    name = tj.split('/')[-2]
    display(Markdown(f'---\n## งาน: {name}'))
    for t in json.load(open(tj, encoding='utf-8')):
        display(Markdown(f"**{t['id']}** — {t['seconds']:.1f} วิ\n> {t['text']}"))
        display(Audio(f"voice_out/{name}/{t['file']}"))

In [ ]:
# 7) รวมทุกงานเป็น zip เดียว (แตกใส่โฟลเดอร์โปรเจกต์บน PC ได้เลย)
!zip -q -r voice_out.zip voice_out
try:
    from google.colab import files
    files.download('voice_out.zip')
except Exception as e:
    print('ดาวน์โหลดอัตโนมัติไม่ได้:', e)
    print('ดึงเองได้ที่แถบ Files 📁 ซ้ายมือ -> voice_out.zip')